# Covid Twitter Project – Master analysis notebook

Questo notebook carica i path con motif e comunità, e produce in modo sistematico:

1. Preprocessing dei dati
2. Profili globali dei motif
3. Distribuzioni dei motif per comunità
4. Matrici di transizione tra comunità (step1, final, last)
5. Misure di persistenza (step1, final, always, last) per comunità e per motif
6. Salvataggio di tutte le tabelle in CSV

Può essere usato sia per la rete originale sia per la rete reversed
(modificando solo `INPUT_CSV` e `RUN_LABEL`).

In [ ]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import jensenshannon, pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list

sns.set(style="whitegrid")

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

DATA_DIR = Path(".")  # oppure il path alla cartella con i CSV

# Cambia qui tra "original" e "reversed" se vuoi
RUN_LABEL = "reversed"   # es. "original" o "reversed"

# Nome file di input (adatta al tuo caso)
INPUT_CSV = DATA_DIR / f"paths_motifs_communities_10e6_{RUN_LABEL}.csv"
# oppure, per l'originale:
# INPUT_CSV = DATA_DIR / "paths_motifs_communities_10e6.csv"

# Comunità principali da tenere (come stringhe)
VALID_COMMS = ["0", "1", "2", "3", "4", "6", "9"]  # 6 principali + "9" (other)
SINK_LABEL = "-1"

print(f"Run label       : {RUN_LABEL}")
print(f"Input file      : {INPUT_CSV}")
print(f"Valid communities: {VALID_COMMS}")

In [ ]:
import pandas as pd

df_orig = pd.read_csv("paths_motifs_communities_10e6_original.csv")
df_rev  = pd.read_csv("paths_motifs_communities_10e6_reversed.csv")

print(len(df_orig), len(df_rev))

In [ ]:
# ---------------------------------------------------------
# Helper: parsing columns
# ---------------------------------------------------------

def ensure_list_column(series):
    """
    Ensure a Series of Python lists, parsing string representations if needed.
    """
    return series.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)


# ---------------------------------------------------------
# Helper: community-level path info
# ---------------------------------------------------------

def get_start_comm(comm_path):
    """Return starting community label (position 0)."""
    return comm_path[0]

def get_next_comm(comm_path, sink_label=SINK_LABEL):
    """Return community at first step (index 1), or sink if missing."""
    return comm_path[1] if len(comm_path) > 1 else sink_label

def get_final_comm(comm_path, sink_label=SINK_LABEL):
    """Return last non-sink community in the path."""
    for c in reversed(comm_path):
        if c != sink_label:
            return c
    return sink_label

def get_last_two_comms(comm_path, sink_label=SINK_LABEL):
    """
    Return (second_last_comm, last_comm) considering only non-sink entries.
    If fewer than 2 non-sink communities, return (None, None).
    """
    non_sink = [c for c in comm_path if c != sink_label]
    if len(non_sink) < 2:
        return None, None
    return non_sink[-2], non_sink[-1]


# ---------------------------------------------------------
# Helper: persistence flags
# ---------------------------------------------------------

def stays_entirely_in_same_comm(comm_path, sink_label=SINK_LABEL):
    """
    True if the path stays entirely inside the starting community,
    ignoring sink values.
    """
    start = comm_path[0]
    for c in comm_path:
        if c == sink_label:
            break  # stop at sink
        if c != start:
            return False
    return True

def last_step_persistent(row):
    """
    True if the last step (second_last_comm -> last_comm)
    stays in the same community.
    """
    a, b = row["second_last_comm"], row["last_comm"]
    if a is None or b is None:
        return False  # puoi usare np.nan se vuoi escluderli
    return a == b


# ---------------------------------------------------------
# Helper: transition matrices
# ---------------------------------------------------------

def transition_matrix(df, from_col, to_col, normalize=True):
    """
    Build a transition matrix between communities:
        rows   = from_col
        columns= to_col

    If normalize=True, each row is turned into probabilities (row-sum=1).
    """
    ctab = pd.crosstab(df[from_col], df[to_col])
    if normalize:
        ctab = ctab.div(ctab.sum(axis=1), axis=0)
    return ctab

In [ ]:
# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------

df = pd.read_csv(INPUT_CSV)

print("Columns:", df.columns.tolist())
print("Number of paths:", len(df))

# Ensure Path_community and (optional) Path are lists
df["Path_community"] = ensure_list_column(df["Path_community"])

# Motif as int
df["Motif"] = df["Motif"].astype(int)

# ---------------------------------------------------------
# 2. Derive community-level columns
# ---------------------------------------------------------

df["start_comm"] = df["Path_community"].apply(get_start_comm)
df["next_comm"] = df["Path_community"].apply(get_next_comm)
df["final_comm"] = df["Path_community"].apply(get_final_comm)

df[["second_last_comm", "last_comm"]] = df["Path_community"].apply(
    lambda p: pd.Series(get_last_two_comms(p))
)

# ---------------------------------------------------------
# 3. Filter to main communities (by start_comm)
# ---------------------------------------------------------

df_filtered = df[df["start_comm"].isin(VALID_COMMS)].copy()
print("Number of paths after filtering:", len(df_filtered))
df_filtered.head()

In [ ]:
# ---------------------------------------------------------
# Global motif profile (normalized frequencies)
# ---------------------------------------------------------

global_motif_counts = df_filtered["Motif"].value_counts().sort_index()
global_motif_profile = global_motif_counts / global_motif_counts.sum()

print("Global motif profile (normalized):")
display(global_motif_profile)

# Optional: simple bar plot
plt.figure(figsize=(6,4), dpi=100)
global_motif_profile.plot(kind="bar")
plt.title(f"Global motif profile – {RUN_LABEL}")
plt.xlabel("Motif")
plt.ylabel("Frequency")
plt.yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Motif distribution by starting community
# ---------------------------------------------------------

motif_by_start = (
    df_filtered
    .groupby(["start_comm", "Motif"])
    .size()
    .unstack(fill_value=0)
)

motif_by_start_norm = motif_by_start.div(motif_by_start.sum(axis=1), axis=0)

print("Motif distribution by starting community (normalized rows):")
display(motif_by_start_norm)

# Save to CSV
out_counts = DATA_DIR / f"motif_distribution_by_start_{RUN_LABEL}.csv"
out_norm   = DATA_DIR / f"motif_distribution_by_start_norm_{RUN_LABEL}.csv"
motif_by_start.to_csv(out_counts)
motif_by_start_norm.to_csv(out_norm)
print("Saved:", out_counts)
print("Saved:", out_norm)

# Optional heatmap
plt.figure(figsize=(8,5), dpi=100)
sns.heatmap(np.log10(motif_by_start_norm+1e-9), annot=False, cmap="viridis")
plt.title(f"Motif distribution by start community – {RUN_LABEL}")
plt.xlabel("Motif")
plt.ylabel("Start community")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# 10. Community distances based on motif profiles
#      (Jensen–Shannon) + dendrogram + heatmap
# ---------------------------------------------------------

# We reuse motif_by_start_norm:
# rows = start_comm, columns = motifs, row-normalized
motif_pivot_norm = motif_by_start_norm.copy()

# Safety: ensure rows are proper probability distributions
motif_pivot_norm = motif_pivot_norm.div(motif_pivot_norm.sum(axis=1), axis=0)

# ---- 10.1 Jensen–Shannon distance matrix between communities ----

# Compute pairwise JS distances with pdist, then expand to square matrix
dist_condensed = pdist(
    motif_pivot_norm.values,
    metric=lambda u, v: jensenshannon(u, v)  # returns JS distance
    #metric=lambda u, v: np.sqrt(np.sum((u - v) ** 2)) # L2 distance
    #metric="cityblock"  # L1 distance
)
dist_matrix = squareform(dist_condensed)

comm_labels = motif_pivot_norm.index.tolist()
dist_js_df = pd.DataFrame(dist_matrix, index=comm_labels, columns=comm_labels)

print("Jensen–Shannon distance matrix between communities:")
display(dist_js_df)

# Save to CSV
out_dist = DATA_DIR / f"community_js_distance_motifs_{RUN_LABEL}.csv"
dist_js_df.to_csv(out_dist)
print("Saved:", out_dist)

# ---- 10.2 Hierarchical clustering (dendrogram) ----

# Linkage on the distance matrix; 'average' is safe for generic distances
Z = linkage(dist_condensed, method="average")

plt.figure(figsize=(6, 4), dpi=100)
dendrogram(Z, labels=comm_labels)
plt.title(f"Communities clustering by motif profile – {RUN_LABEL}")
plt.ylabel("Jensen–Shannon distance")
plt.tight_layout()
plt.show()

# ---- 10.3 Heatmap of the JS distances, ordered by clustering ----

# Get the leaf order from the dendrogram
leaf_order = leaves_list(Z)
ordered_labels = [comm_labels[i] for i in leaf_order]

dist_js_ordered = dist_js_df.loc[ordered_labels, ordered_labels]

plt.figure(figsize=(6, 5), dpi=100)
sns.heatmap(dist_js_ordered, annot=False, cmap="mako", square=True)
plt.title(f"Community JS distances (ordered by clustering) – {RUN_LABEL}")
plt.xlabel("Community")
plt.ylabel("Community")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Aggregated transition matrices (all motifs together)
# ---------------------------------------------------------

# Drop paths where second_last_comm / last_comm are None for last-step
df_last = df_filtered.dropna(subset=["second_last_comm", "last_comm"]).copy()

transition_step1 = transition_matrix(df_filtered, "start_comm", "next_comm", normalize=True)
transition_final = transition_matrix(df_filtered, "start_comm", "final_comm", normalize=True)
transition_last  = transition_matrix(df_last, "second_last_comm", "last_comm", normalize=True)

print("First-step transition matrix (start_comm → next_comm):")
display(transition_step1)

print("Final-step transition matrix (start_comm → final_comm):")
display(transition_final)

print("Last-step transition matrix (second_last_comm → last_comm):")
display(transition_last)

# Save to CSV
transition_step1.to_csv(DATA_DIR / f"transition_step1_allmotifs_{RUN_LABEL}.csv")
transition_final.to_csv(DATA_DIR / f"transition_final_allmotifs_{RUN_LABEL}.csv")
transition_last.to_csv(DATA_DIR / f"transition_last_allmotifs_{RUN_LABEL}.csv")

In [ ]:
def plot_transition_heatmap(mat, title):
    plt.figure(figsize=(6,5), dpi=100)
    sns.heatmap(mat, annot=False, cmap="Blues")
    plt.title(title)
    plt.xlabel("To community")
    plt.ylabel("From community")
    plt.tight_layout()
    plt.show()

plot_transition_heatmap(transition_step1, f"First-step transitions – {RUN_LABEL}")
plot_transition_heatmap(transition_final, f"Final-step transitions – {RUN_LABEL}")
plot_transition_heatmap(transition_last,  f"Last-step transitions – {RUN_LABEL}")

In [ ]:
# ---------------------------------------------------------
# Transition matrices by motif
# ---------------------------------------------------------

motifs = sorted(df_filtered["Motif"].unique())

step1_mats = {}
final_mats = {}
last_mats  = {}

for m in motifs:
    df_m = df_filtered[df_filtered["Motif"] == m].copy()
    df_m_last = df_m.dropna(subset=["second_last_comm", "last_comm"])
    
    if len(df_m) == 0:
        step1_mats[m] = pd.DataFrame()
        final_mats[m] = pd.DataFrame()
        last_mats[m]  = pd.DataFrame()
        continue
    
    # step1
    ctab_step1 = pd.crosstab(df_m["start_comm"], df_m["next_comm"])
    step1_mats[m] = ctab_step1.div(ctab_step1.sum(axis=1), axis=0)
    
    # final
    ctab_final = pd.crosstab(df_m["start_comm"], df_m["final_comm"])
    final_mats[m] = ctab_final.div(ctab_final.sum(axis=1), axis=0)
    
    # last
    if len(df_m_last) > 0:
        ctab_last = pd.crosstab(df_m_last["second_last_comm"], df_m_last["last_comm"])
        last_mats[m] = ctab_last.div(ctab_last.sum(axis=1), axis=0)
    else:
        last_mats[m] = pd.DataFrame()

# Quick sanity check for one motif
example_motif = motifs[0]
print(f"Example motif {example_motif} – step1 matrix:")
display(step1_mats[example_motif])

print(f"Example motif {example_motif} – final matrix:")
display(final_mats[example_motif])

print(f"Example motif {example_motif} – last-step matrix:")
display(last_mats[example_motif])

In [ ]:
for m in motifs:
    if not step1_mats[m].empty:
        step1_mats[m].to_csv(DATA_DIR / f"transition_step1_motif{m}_{RUN_LABEL}.csv")
    if not final_mats[m].empty:
        final_mats[m].to_csv(DATA_DIR / f"transition_final_motif{m}_{RUN_LABEL}.csv")
    if not last_mats[m].empty:
        last_mats[m].to_csv(DATA_DIR / f"transition_last_motif{m}_{RUN_LABEL}.csv")

print("Saved motif-specific transition matrices.")

In [ ]:
# ---------------------------------------------------------
# Persistence flags on df_filtered
# ---------------------------------------------------------

# First-step persistence: start_comm == next_comm
df_filtered["step1_same_comm"] = df_filtered["start_comm"] == df_filtered["next_comm"]

# Final persistence: start_comm == final_comm
df_filtered["final_same_comm"] = df_filtered["start_comm"] == df_filtered["final_comm"]

# Always-inside persistence
df_filtered["always_inside"] = df_filtered["Path_community"].apply(stays_entirely_in_same_comm)

# Last-step persistence: second_last_comm == last_comm
df_filtered["last_step_same_comm"] = df_filtered.apply(last_step_persistent, axis=1)

# ---------------------------------------------------------
# Group by (Motif, start_comm)
# ---------------------------------------------------------

grouped = df_filtered.groupby(["Motif", "start_comm"])

persistence_step1   = grouped["step1_same_comm"].mean().rename("persistence_step1")
persistence_final   = grouped["final_same_comm"].mean().rename("persistence_final")
persistence_always  = grouped["always_inside"].mean().rename("persistence_always")
persistence_last    = grouped["last_step_same_comm"].mean().rename("persistence_last")

pers_motif_comm = pd.concat(
    [persistence_step1, persistence_final, persistence_always, persistence_last],
    axis=1
).sort_index()

print("Full persistence table by (Motif, start_comm):")
display(pers_motif_comm)

# Save
out_pers = DATA_DIR / f"persistence_by_motif_comm_{RUN_LABEL}.csv"
pers_motif_comm.to_csv(out_pers)
print("Saved:", out_pers)

In [ ]:
# ---------------------------------------------------------
# Persistence aggregated over communities (per motif only)
# ---------------------------------------------------------

grouped_motif = df_filtered.groupby("Motif")

p_step1_m = grouped_motif["step1_same_comm"].mean().rename("persistence_step1")
p_final_m = grouped_motif["final_same_comm"].mean().rename("persistence_final")
p_always_m = grouped_motif["always_inside"].mean().rename("persistence_always")
p_last_m = grouped_motif["last_step_same_comm"].mean().rename("persistence_last")

pers_motif_global = pd.concat(
    [p_step1_m, p_final_m, p_always_m, p_last_m],
    axis=1
).sort_index()

print("Persistence per motif (aggregated over communities):")
display(pers_motif_global)

out_pers_global = DATA_DIR / f"persistence_by_motif_global_{RUN_LABEL}.csv"
pers_motif_global.to_csv(out_pers_global)
print("Saved:", out_pers_global)